# Step 3: baseline training on Colab

Runtime > Change runtime type > **GPU** (T4 is enough). Run the cells top to bottom.
After a disconnect: rerun cells 1-5, skip the smoke test and the full run, and run **Resume**.

Weights and the MLflow DB are saved to Google Drive (`MyDrive/aerial-edge`); the dataset and the
live SQLite DB stay on the VM disk, which is faster and safer for SQLite.

In [1]:
# 1. GPU check
!nvidia-smi

Sat Sep 26 14:13:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Mount Google Drive (weights and the MLflow backup go here)
from google.colab import drive

drive.mount("/content/drive")
D = "/content/drive/MyDrive/aerial-edge"
!mkdir -p {D}

Mounted at /content/drive


In [3]:
# 3. Get the code. For a private repo use https://<token>@github.com/...
REPO_URL = "https://github.com/mttrin93/aerial-edge.git"  # change to your repo
REPO = "/content/aerial-edge"
!test -d {REPO} || git clone {REPO_URL} {REPO}
%cd {REPO}
!git pull --ff-only

Cloning into '/content/aerial-edge'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 37 (delta 7), reused 37 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 180.14 KiB | 36.03 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/aerial-edge
Already up to date.


In [4]:
# 4. Install: keep Colab's CUDA torch, add only what training needs.
# PYTHONPATH instead of `pip install -e .`, so the package's Python pin does not matter here.
!pip install -q ultralytics==8.4.163 mlflow==3.16.1
%env PYTHONPATH=/content/aerial-edge/src
%env MLFLOW_TRACKING_URI=sqlite:////content/mlflow.db

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 438.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [5]:
# 5. Data: VisDrone2019-DET (~2 GB) to the VM disk, a few minutes per session
!yolo settings datasets_dir=/content/datasets
!python scripts/download_data.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ Updated 'datasets_dir=/content/datasets'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.8",
  "datasets_dir": "/content/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "569f3ba64b326db489132663f79cd37279811de477381b83ac131e6cdd129cbb",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "mlflow": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/usage/settings

WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/content/datasets/VisD

## Smoke test

One epoch on 1% of the data, logged to a separate throwaway DB and folder. Checks that data,
GPU, MLflow and the Drive backup all work before the long run.

In [6]:
!MLFLOW_TRACKING_URI=sqlite:////content/smoke.db python scripts/train.py epochs=1 fraction=0.01 \
    project=/content/smoke --backup-mlflow /content/smoke.db {D}/smoke.db

MLflow tracking URI: sqlite:////content/smoke.db
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.01, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_640, n

## Full run

100 epochs, settings from `configs/train.yaml`. Weights go to `{D}/runs/yolo26n_640/weights`.

In [7]:
!python scripts/train.py project={D}/runs --backup-mlflow /content/mlflow.db {D}/mlflow.db

MLflow tracking URI: sqlite:////content/mlflow.db
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_640,

## Resume (only after a disconnect)

Restore the MLflow DB from Drive, then continue from `last.pt`.

In [ ]:
!cp {D}/mlflow.db /content/mlflow.db
!python scripts/train.py --resume {D}/runs/yolo26n_640/weights/last.pt \
    --backup-mlflow /content/mlflow.db {D}/mlflow.db

## Results

Last epochs from `results.csv`. Download `best.pt` and `mlflow.db` from Drive to the laptop:
`best.pt` into `models/`, `mlflow.db` into `mlflow/`.

In [8]:
import pandas as pd

df = pd.read_csv(f"{D}/runs/yolo26n_640/results.csv")
df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail()

,epoch,metrics/mAP50(B),metrics/mAP50-95(B)
95,96,0.33875,0.18485
96,97,0.33912,0.18564
97,98,0.33896,0.18548
98,99,0.33907,0.18545
99,100,0.33906,0.18555
